
Note book Extract Script - Thu thập dữ liệu từ Binance API

Chức năng:
1. Extract Symbols: Lấy thông tin 50 coins từ /exchangeInfo, lưu thẳng vào data/raw 
2. [TEST] Extract Klines: Lấy dữ liệu nến 1 phút từ /klines
3. [TEST] Extract Ticker 24h + Best Bid/Ask: Lấy thống kê 24h từ /ticker/24hr và /ticker/bookTicker
4. [TEST] Extract Order Book Snapshot: Lấy snapshot từ /depth
(Test trước 10 symbol, lưu tạm tại notebooks/01_data_exploration/data/raw/)
Output:
   - data/raw/symbols.json
   - data/raw/{SYMBOL}.csv (50 files)
   - data/raw/ticker_24h.csv
   - data/raw/order_book_snapshot.csv

Sử dụng: python scripts/extract.py --start-date 2023-01-01 --end-date 2026-01-01


In [ ]:
# 1. Extract Symbols: Lấy thông tin 50 coins từ /exchangeInfo:
import csv
import json
import os
from typing import List, Dict
import logging
from pathlib import Path

import requests

BASE_URL = "https://api.binance.com"

SYMBOLS: List[str] = [
    "BTCUSDT", "ETHUSDT", "BNBUSDT", "SOLUSDT", "XRPUSDT",
    "DOGEUSDT", "ADAUSDT", "TRXUSDT", "LINKUSDT", "MATICUSDT",
    "AVAXUSDT", "TONUSDT", "SHIBUSDT", "XLMUSDT", "BCHUSDT",
    "DOTUSDT", "UNIUSDT", "LTCUSDT", "HBARUSDT", "PEPEUSDT",
    "NEARUSDT", "APTUSDT", "ICPUSDT", "ETCUSDT", "STXUSDT",
    "RENDERUSDT", "CROUSDT", "ATOMUSDT", "VETUSDT", "ARBUSDT",
    "INJUSDT", "IMXUSDT", "OPUSDT", "GRTUSDT", "THETAUSDT",
    "FILUSDT", "ARUSDT", "MKRUSDT", "WIFUSDT", "RUNEUSDT",
    "FTMUSDT", "ALGOUSDT", "FLOWUSDT", "XTZUSDT", "AXSUSDT",
    "SANDUSDT", "MANAUSDT", "NEOUSDT", "EOSUSDT", "AAVEUSDT",
]

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

def _parse_symbols(payload: dict) -> List[Dict[str, str]]:
    results = []
    for item in payload.get("symbols", []):
        results.append(
            {
                "symbol": item.get("symbol"),
                "base_asset": item.get("baseAsset"),
                "quote_asset": item.get("quoteAsset"),
                "status": item.get("status"),
            }
        )
    return results

def get_exchange_info(symbols: List[str]) -> List[Dict[str, str]]:
    params = {"symbols": json.dumps(symbols)}
    try:
        response = requests.get(f"{BASE_URL}/api/v3/exchangeInfo", params=params, timeout=30)
        response.raise_for_status()
        return _parse_symbols(response.json())
    except requests.HTTPError as exc:
        if exc.response is not None and exc.response.status_code == 400:
            full = requests.get(f"{BASE_URL}/api/v3/exchangeInfo", timeout=30)
            full.raise_for_status()
            payload = full.json()
            valid = {item.get("symbol") for item in payload.get("symbols", [])}
            filtered = [s for s in symbols if s in valid]
            return [item for item in _parse_symbols(payload) if item.get("symbol") in set(filtered)]
        raise

def save_csv(rows: List[Dict[str, str]], path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["symbol", "base_asset", "quote_asset", "status"])
        writer.writeheader()
        writer.writerows(rows)

def save_json(rows: List[Dict[str, str]], path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for _ in range(10):
        if (current / "data" / "raw").exists():
            return current
        if current.parent == current:
            break
        current = current.parent
    return start.resolve()

def main() -> None:
    rows = get_exchange_info(SYMBOLS)

    start_dir = Path.cwd()
    project_root = find_project_root(start_dir)
    raw_dir = project_root / "data" / "raw"

    csv_path = raw_dir / "symbols.csv"
    json_path = raw_dir / "symbols.json"

    save_csv(rows, str(csv_path))
    save_json(rows, str(json_path))

    found = {r.get("symbol") for r in rows}
    expected = set(SYMBOLS)
    missing = sorted(expected - found)

    logger.info("Saved %s symbols to: %s", len(rows), csv_path)
    logger.info("Saved %s symbols to: %s", len(rows), json_path)
    if missing:
        logger.warning("Missing symbols: %s", ", ".join(missing))
    else:
        logger.info("All symbols returned.")

if __name__ == "__main__":
    main()

[TEST] Extract Klines: Lấy dữ liệu nến 1 phút từ /klines
- Lấy data 10 symbols trong folder data mẫu
- Tạm thời lấy dữ liệu 1 tháng trước thay vì 3 năm 

In [ ]:
# 2. Extract Klines: Tải dữ liệu nến 1 phút từ Binance Data Vision
import io
import zipfile
from datetime import datetime
from dateutil.relativedelta import relativedelta
import pandas as pd

BINANCE_DATA_VISION_URL = "https://data.binance.vision/data/spot/monthly/klines/{symbol}/1m/{symbol}-1m-{year}-{month:02d}.zip"

# Đường dẫn lưu file
RAW_DATA_DIR = Path("./data/raw")
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Đọc danh sách symbols từ file CSV
symbols_df = pd.read_csv(RAW_DATA_DIR / "symbols.csv")
symbols_list = symbols_df["symbol"].tolist()

def download_klines(symbol: str, year: int, month: int) -> pd.DataFrame | None:
    url = BINANCE_DATA_VISION_URL.format(symbol=symbol, year=year, month=month)
    logger.info(f"Downloading: {url}")
    
    try:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        
        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
            csv_filename = z.namelist()[0]
            with z.open(csv_filename) as csv_file:
                df = pd.read_csv(
                    csv_file,
                    header=None,
                    names=[
                        "open_time", "open", "high", "low", "close", "volume",
                        "close_time", "quote_volume", "trades", 
                        "taker_buy_base", "taker_buy_quote", "ignore"
                    ]
                )

                # Parse tối thiểu
                df["open_time"] = pd.to_datetime(df["open_time"], unit="ms")
                df["close_time"] = pd.to_datetime(df["close_time"], unit="ms")
                df["symbol"] = symbol

                return df
                
    except requests.HTTPError as e:
        logger.warning(f"HTTP Error for {symbol} ({year}-{month:02d}): {e}")
        return None
    except Exception as e:
        logger.error(f"Error downloading {symbol}: {e}")
        return None


def extract_klines(symbols: list, months_back: int = 1) -> dict:
    """
    Tải dữ liệu klines cho danh sách symbols.
    
    Args:
        symbols: Danh sách trading pairs
        months_back: Số tháng cần tải (tính từ tháng trước)
    
    Returns:
        Dict với key là symbol và value là DataFrame
    """
    results = {}
    
    # Tính tháng bắt đầu (tháng trước)
    end_date = datetime.now() - relativedelta(months=1)
    
    for symbol in symbols:
        all_data = []
        
        for i in range(months_back):
            target_date = end_date - relativedelta(months=i)
            year = target_date.year
            month = target_date.month
            
            df = download_klines(symbol, year, month)
            if df is not None:
                all_data.append(df)
                logger.info(f"✓ {symbol}: {year}-{month:02d} - {len(df):,} records")
        
        if all_data:
            # Gộp tất cả các tháng
            combined_df = pd.concat(all_data, ignore_index=True)
            combined_df = combined_df.sort_values("open_time").reset_index(drop=True)
            
            # Lưu file CSV
            output_path = RAW_DATA_DIR / f"{symbol}.csv"
            combined_df.to_csv(output_path, index=False)
            logger.info(f"Saved {symbol} to {output_path} ({len(combined_df):,} records)")
            
            results[symbol] = combined_df
        else:
            logger.warning(f"✗ No data for {symbol}")
    
    return results


# Chạy extract với 1 tháng dữ liệu (tháng 01/2026)
logger.info(f"Starting klines extraction for {len(symbols_list)} symbols...")
logger.info(f"Symbols: {symbols_list}")

klines_data = extract_klines(symbols_list, months_back=6)

# Thống kê kết quả
logger.info("=" * 50)
logger.info(f"Extraction completed!")
logger.info(f"Successfully downloaded: {len(klines_data)}/{len(symbols_list)} symbols")

total_records = sum(len(df) for df in klines_data.values())
logger.info(f"Total records: {total_records:,}")